# Public-source note

Paths have been sanitized. Notebook outputs are temporarily cleared in this draft while the workflow is being reorganized. For the final repository, the cleaned notebooks will be re-run and their output cells retained; any machine-specific paths printed in outputs will be removed or replaced with portable labels.


In [ ]:

library(DESeq2)
library(readr)
library(dplyr)
library(pheatmap)
library(ggplot2)
library(stringr)
library(factoextra)
library(tidyr)
library(ggpubr)
# library(btools)


In [ ]:
readspergene_dir="/path/to/cebp_seq/bulk_rna/5_readspergene_out"
output_dir="/path/to/CEBPB_seq/output/bulk_rna/Cluster_all_starting_cell_types"
plot_dir="/path/to/CEBPB_seq/plots/bulk_rna/Cluster_all_starting_cell_types"

In [ ]:
samples <- list.files(path = readspergene_dir, 
                full.names = FALSE, 
                pattern="ReadsPerGene.out.tab")


# Ensure the object is treated as character (not factor)
samples <- as.character(samples)

In [ ]:
samples

In [ ]:
groups_to_keep <- c(
    "MEF_Rep",
    "MEF_d2_OSKM_Rep",
    "MEF_d2_OSKM_mCEBPA",
    "MEF_d2_OSKM_mCEBPB",
    "MEF_d12_OSKM",
    "MEF_d12_OSKM_mCEBPAflag",
    "MEF_d12_OSKM_mCEBPBflag",
    "NSC_rep",
    "NSC_d2_OSKM_mCEBPAflag_rep",
    "NSC_d2_OSKM_mCEBPBflag_rep",
    "NSC_d12_OSKM_mCEBPAflag",
    "NSC_d12_OSKM_mCEBPBflag",
    # "NSC_d12_mCEBPAflag",
    # "NSC_d12_mCEBPBflag",
    'ESC_Rep',
    'ESC_d1_hCEBPA_Rep',
    'ESC_d2_hCEBPA_Rep',
    "Bcell_Rep",
    "Bcell_Cebpa18h_Rep",
    "Bcell_Cebpa18h_OKSMd1_Rep",
    "Bcell_Cebpa18h_OKSMd2_Rep")


In [ ]:
# Keep only samples containing any of these substrings
samples_filt <- samples[grepl(paste(groups_to_keep, collapse = "|"), samples)]

In [ ]:
samples_filt

In [ ]:
length(samples_filt)

In [ ]:
length(groups_to_keep)

In [ ]:

# Clean up sample names
clean_names <- gsub('"', '', samples_filt)         # remove any stray double-quote characters
clean_names <- trimws(clean_names)            # remove leading/trailing whitespace
clean_names <- sub("^(.*[Rr]ep[0-9]+).*",     # keep everything up to "repX" or "RepX"
                   "\\1",
                   clean_names)
clean_names <- sub("rep([0-9]+)",             # normalize to "RepX" (capital R)
                   "Rep\\1",
                   clean_names,
                   ignore.case = TRUE)

# Assign cleaned sample names to the files vector
# -> now names(samples) will be "Sample_Rep1", "Sample_Rep2", etc.
names(samples_filt) <- clean_names

In [ ]:
samples_filt

In [ ]:
# Read gene lengths table (tab-delimited) into R
gene.lengths <- read_delim("/path/to/refdata/igenomes/index/GRCm39/GRCm39_gencodevM33_geneLengths.txt", "\t",
                           trim_ws = TRUE, escape_double = TRUE)

In [ ]:
head(gene.lengths)

In [ ]:
gene.lengths %>%
  filter(gene_id == "Tp63")

In [ ]:
gene.lengths %>%
  filter(gene_name == "Rxra")

In [ ]:
for (fi in seq_along(samples_filt)) {
  fname <- samples_filt[fi]

  dir_to_use <- if (any(startsWith(fname, c("Beige", "WAT", "BAT")))) {
    Adipo_GSE198107_readspergene_dir
  } else {
    readspergene_dir
  }

  data <- read_delim(
    file.path(dir_to_use, fname),
    "\t",
    trim_ws = TRUE,
    escape_double = TRUE,
    col_names = c("gene_id", "unstranded", "firststrand", "secondstrand"),
    skip = 4
  )

  cor_first  <- cor(data$unstranded, data$firststrand)
  cor_second <- cor(data$unstranded, data$secondstrand)

  # decide which strand to use
  strand_to_use <- if (cor_first > 0.9 && cor_second > 0.9) {
    "unstranded"
  } else if (cor_first > cor_second) {
    "firststrand"
  } else {
    "secondstrand"
  }

  message(sprintf("[%s] Using %s (cor_first=%.3f, cor_second=%.3f)",
                  fname, strand_to_use, cor_first, cor_second))

  # you can now use: data[[strand_to_use]] for downstream analysis
}


In [ ]:
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(purrr)
})

choose_dir <- function(fname) {
  if (any(startsWith(fname, c("Beige", "WAT", "BAT")))) {
    Adipo_GSE198107_readspergene_dir
  } else if (startsWith(fname, "chronis_cell")) {
    chronis_cell_readspergene_dir
  } else {
    readspergene_dir
  }
}

pick_strand <- function(df) {
  cor_first  <- cor(df$unstranded, df$firststrand)
  cor_second <- cor(df$unstranded, df$secondstrand)
  if (cor_first > 0.9 && cor_second > 0.9) "unstranded" else if (cor_first > cor_second) "firststrand" else "secondstrand"
}

id_from_fname <- function(fname) sub("\\.out\\.tab$", "", fname)

parts <- vector("list", length(samples_filt))

for (fi in seq_along(samples_filt)) {
  fname <- samples_filt[fi]
  dir_i <- choose_dir(fname)

  df <- read_delim(
    file.path(dir_i, fname),
    "\t",
    trim_ws = TRUE,
    escape_double = TRUE,
    col_names = c("gene_id", "unstranded", "firststrand", "secondstrand"),
    skip = 4
  )

  strand <- pick_strand(df)

  data <- df %>%
    select(gene_id, all_of(strand))

  # keep this exact renaming style, with a safe fallback
  colnames(data)[2] <- if (!is.null(names(samples_filt)) && nzchar(names(samples_filt)[fi])) {
    names(samples_filt)[fi]
  } else {
    id_from_fname(fname)
  }

  parts[[fi]] <- data
}

# combine all by gene_id
sample_data <- reduce(parts, full_join, by = "gene_id")


In [ ]:
head(sample_data)

In [ ]:
# Attach gene_name to counts by gene_id
sample_data <- left_join(sample_data,
                         dplyr::select(gene.lengths, gene_id, gene_name),
                         by = "gene_id")

# Identify count columns programmatically
count_cols <- setdiff(names(sample_data), c("gene_id", "gene_name"))

# Compute total counts per gene across all samples
sample_data <- sample_data %>%
  mutate(total = rowSums(across(all_of(count_cols), as.numeric), na.rm = TRUE))

# For gene symbols that map to multiple gene_ids, keep the row with the highest total
# top_n is deprecated, use slice_max
sample_data <- sample_data %>%
  group_by(gene_name) %>%
  slice_max(order_by = total, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  select(-total, -gene_id)

In [ ]:
head(sample_data)

In [ ]:
nrow(sample_data)

In [ ]:
# number of rows with any NA
sum(!complete.cases(sample_data))


In [ ]:
cat("Rows with NA:", sum(!complete.cases(sample_data)), "\n")
cat("Cols with NA:", sum(colSums(is.na(sample_data)) > 0), "\n")


In [ ]:

# Identify count columns programmatically
count_cols <- setdiff(names(sample_data), c("gene_id", "gene_name"))

# Build numeric matrix with gene_name as rownames
data_mat <- as.matrix(select(sample_data, all_of(count_cols)))
rownames(data_mat) <- sample_data$gene_name


# Define your metadata
meta.data <- data.frame(sample_rep = colnames(data_mat))

# Remove replicate part (_Rep1, _Rep2, etc.) to get the biological sample ID
meta.data$sample <- str_replace(meta.data$sample_rep, "_Rep[0-9]+$", "")

# Optional: extract replicate number as a separate column
meta.data$replicate <- str_extract(meta.data$sample_rep, "Rep[0-9]+")

# Make sure columns and rows of metadata are in the exact same order
all(meta.data$sample_rep == colnames(data_mat))


#    Filter out very lowly expressed genes
#    Keep genes with total counts > 2 across all samples
data_mat <- data_mat[rowSums(data_mat) > 2, , drop = FALSE]

#    Make sure metadata rows match columns of the count matrix
stopifnot(identical(meta.data$sample_rep, colnames(data_mat)))

#    Use a clean design column
#    Convert 'sample' to a factor so DESeq2 treats it as categorical
meta.data$sample <- factor(meta.data$sample)

In [ ]:
meta.data

In [ ]:
#   Build DESeq2 dataset and run the pipeline
dds <- DESeqDataSetFromMatrix(countData = round(data_mat),
                              colData   = meta.data,
                              design    = ~ sample)

dds <- DESeq(dds)



In [ ]:
## DESeq2_normalized_counts
norm_counts <- counts(dds, normalized=T)
# write.csv(norm_counts, file.path( output_dir, paste0("Starting_cell_types_DESeq2_normalized_counts.csv")), quote=F)


          
### VST normalized counts
vst_counts <- vst(dds, blind=FALSE)
vst_counts_df <- assay(vst_counts)
# write.csv(vst_counts_df, file.path( output_dir,paste0("Starting_cell_types_DESeq2_VST_counts.csv")), quote=F)


In [ ]:
library(dplyr)
library(readr)
library(stringr)
library(tidyr)

## 1) Build a clean gene_name -> gene_length_bp lookup
# try common length column names in your file
len_col_candidates <- c("gene_length", "length", "length_bp", "bp", "tx_length", "geneLen")
len_col <- intersect(len_col_candidates, names(gene.lengths))
stopifnot(length(len_col) >= 1)
len_col <- len_col[1]  # take the first match

length_tbl <- gene.lengths |>
  select(gene_name, gene_length_bp = all_of(len_col)) |>
  filter(!is.na(gene_name), !is.na(gene_length_bp)) |>
  group_by(gene_name) |>
  summarize(gene_length_bp = median(as.numeric(gene_length_bp), na.rm = TRUE), .groups = "drop")

# lengths aligned to your count matrix rows
len_vec_bp <- length_tbl$gene_length_bp[match(rownames(data_mat), length_tbl$gene_name)]
names(len_vec_bp) <- rownames(data_mat)

# drop genes with missing or zero length
keep_len <- !is.na(len_vec_bp) & len_vec_bp > 0
if (!all(keep_len)) {
  message("Dropping ", sum(!keep_len), " genes with missing or zero length before TPM")
}
data_mat_len <- data_mat[keep_len, , drop = FALSE]
len_vec_kb   <- (len_vec_bp[keep_len]) / 1000

## 2) RPK and per sample scaling
# RPK = counts divided by gene length in kilobases
rpk <- sweep(data_mat_len, 1, len_vec_kb, "/")

# per sample scaling factor is sum of RPKs divided by one million
scale_factors <- colSums(rpk, na.rm = TRUE) / 1e6

## 3) TPM
tpm <- sweep(rpk, 2, scale_factors, "/")

## 4) Optional sanity checks
stopifnot(all(abs(colSums(tpm) - 1e6) < 1e-6 | is.na(colSums(tpm))))  # each column sums to one million

## 5) Save
# write.csv(as.data.frame(tpm), file.path(output_dir, paste0(cell_type, "_TPM.csv")), quote = FALSE)


In [ ]:
# 1) Map replicate columns to base sample names
rep_map <- tibble(
  col = colnames(tpm),
  sample = str_replace(colnames(tpm), "_Rep[0-9]+$", "")
)


# 2) Average replicates to sample means
tpm_avg <- as.data.frame(tpm) |>
  tibble::rownames_to_column("gene") |>
  pivot_longer(-gene, names_to = "col", values_to = "TPM") |>
  left_join(rep_map, by = "col") |>
  tidyr::drop_na(sample) |>
  group_by(gene, sample) |>
  summarize(TPM = mean(TPM, na.rm = TRUE), .groups = "drop") |>
  pivot_wider(names_from = sample, values_from = TPM)

In [ ]:
rep_map

In [ ]:
head(tpm_avg)

In [ ]:
head(tpm_avg)

In [ ]:
nrow(tpm_avg)

In [ ]:
figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4_V2/"

In [ ]:
# Publication-facing motif annotation table.
# This compact TSV replaces the two historical working annotation files.
manual_ordered_joined_df <- read.delim(
  "fig5c_motif_annotations.tsv",
  header = TRUE,
  sep = "\t",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# Aliases retained so downstream cells from the original analysis remain readable.
kc_df <- manual_ordered_joined_df
manual_ordered_df <- manual_ordered_joined_df

colnames(manual_ordered_joined_df)


In [ ]:
head(kc_df)

In [ ]:
# Motif annotations are loaded from fig5c_motif_annotations.tsv above.


In [ ]:
# Keep the full publication-facing annotation table; no additional column reduction is required.


In [ ]:
# manual_ordered_joined_df is already loaded from fig5c_motif_annotations.tsv.


In [ ]:
# manual_ordered_df <- read.table(
#   paste("Finemo_PrATAC_patterns_0.05_tomtom_manual_match_manual_selected_with_KC_motifs_V4.csv",sep=""),
#   header = TRUE,
#   sep = ",",
#     # row.names = FALSE,
#   stringsAsFactors = FALSE
# )
# head(manual_ordered_df)

In [ ]:
manual_ordered_joined_df$cluster

In [ ]:
length(unique(manual_ordered_joined_df$pattern))

In [ ]:
## keep only the rows where to_plot is 1
# manual_ordered_df <- manual_ordered_joined_df[manual_ordered_joined_df$to_plot == 1, ]

In [ ]:
length(unique(manual_ordered_joined_df$pattern))

In [ ]:
desired_order <- c(
    "MEF",
    "MEF_d2_OSKM",
    "MEF_d2_OSKM_mCEBPAflag",
    "MEF_d2_OSKM_mCEBPBflag",
    "MEF_d12_OSKM",
    "MEF_d12_OSKM_mCEBPAflag",
    "MEF_d12_OSKM_mCEBPBflag",
    "NSC",
    "NSC_d2_OSKM_mCEBPAflag", 
    "NSC_d2_OSKM_mCEBPBflag",
    "NSC_d12_OSKM_mCEBPAflag",
    "NSC_d12_OSKM_mCEBPBflag",
    # "NSC_d12_mCEBPAflag",
    # "NSC_d12_mCEBPBflag",
    # 'ESC_Rep',
    "ESC",  
    "ESC_d1_hCEBPA", 
    "ESC_d2_hCEBPA",
    "Bcell",
    "Bcell_Cebpa18h",
    "Bcell_Cebpa18h_OKSMd1",
    "Bcell_Cebpa18h_OKSMd2")




In [ ]:
desired_order_rep_cols <- rep_map |>
  filter(sample %in% desired_order) |>
  mutate(sample = factor(sample, levels = desired_order)) |>
  arrange(sample, col) |>
  pull(col)

In [ ]:
head(desired_order_rep_cols)

In [ ]:
length(unique(manual_ordered_joined_df$Formula_pick))

In [ ]:
missing_genes1 <- setdiff(
  toupper(manual_ordered_joined_df$Formula_pick),
  toupper(tpm_avg$gene)
)

missing_genes1

In [ ]:
# gene_dict1 <- list(
# 'TEAD X2 (TANDEM)' = c("TEAD4"),
# 'FOS+JUN' =c('JUN'),
# 'TWST1' =c('TWIST1'),
# 'TEAD::AP-1' =c('TEAD1'),
# 'SRF / CARG' =c('SRF'),
# 'CRE::C/EBP' =c('CEBPB'),
# 'C/EBP::AP-1' =c('CEBPB'),
# 'C/EBP::ETS' =c('CEBPB'),
# 'NF-Y (NFYC)' =c('NFYC'),
# 'RFX3_RFX_2' =c('RFX3'),
# 'OCT4/SOX2' =c('POU5F1'),
# 'ZNF143' =c('Zfp143'),
# 'TEAD::C/EBP' =c('Cebpb'),
# 'SP/KLF::ETS' =c('SP1'),
# 'ETS::NF-KB' =c('RELA'),
# 'NF-KB'= c('Nfkb1'),
# 'MEF2' =c('MEF2C'),
# 'MITF/SREBF1' =c('SREBF1'),
# 'IRF::ETS (EICE)' =c('Irf8')
# )


In [ ]:
gene_dict1 <- list(
  "HXA9"       = "HOXA9",
  "FOS+JUN" = c("FOS"),
  "FOSB/ATF3" = c("FOSB"),
   "TWST1"      = "TWIST1",
  "FOXO1/FOXO3" = c("FOXO1"),
  "NR1H3+RXRA" = c("NR1H3"),
  "ZFP28/JDP2" = c("ZFP28"),
  "TEAD1/TEAD3" = c("TEAD1"),
  "FOSL1+JUN" = c("FOSL1"),
  "RFX3_RFX_2" = c("RFX3"),
  "OCT4/SOX2" = c("POU5F1"),
  "OCT4" = c("POU5F1"),
  "STA5A/STA5B" = c("STAT5A"),
  "SPI1/SPIB" = c("SPI1")
)

In [ ]:
# manual_ordered_df$Selected_TF

In [ ]:
# manual_ordered_df$Selected_TF <- trimws(manual_ordered_df$Selected_TF)
# manual_ordered_df$Selected_TF<- ifelse(
#   manual_ordered_df$Selected_TF %in% names(gene_dict1),
#   gene_dict1[manual_ordered_df$Selected_TF],
#   manual_ordered_df$Selected_TF
# )
# manual_ordered_df$Selected_TF[grepl("Rxra", manual_ordered_df$Selected_TF)]<- "Rxra"

In [ ]:
# gene_dict1_vec <- unlist(gene_dict1)

# x <- trimws(as.character(manual_ordered_joined_df$Formula_pick))

# # Find dictionary entries ignoring capitalization
# m <- match(
#   toupper(x),
#   toupper(names(gene_dict1_vec))
# )

# # Replace matched values
# hit <- !is.na(m)

# x[hit] <- unname(gene_dict1_vec[m[hit]])

# manual_ordered_joined_df$Rep_TF_for_RNA <- x

In [ ]:
# manual_ordered_df$Selected_TF <- trimws(as.character(manual_ordered_df$Selected_TF))

# gene_dict1_vec <- unlist(gene_dict1)

# idx <- manual_ordered_df$Selected_TF %in% names(gene_dict1_vec)

# manual_ordered_df$Selected_TF[idx] <-
#   gene_dict1_vec[manual_ordered_df$Selected_TF[idx]]

In [ ]:
grep(
  "^Zfp143$",
  tpm_avg$gene,
  value = TRUE,
  ignore.case = TRUE
)

In [ ]:
missing_genes1 <- setdiff(
  toupper(manual_ordered_joined_df$Formula_pick),
  toupper(tpm_avg$gene)
)

missing_genes1

In [ ]:
missing_genes1 <- setdiff(
  toupper(manual_ordered_joined_df$Formula_pick),
  toupper(tpm_avg$gene)
)

missing_genes1


In [ ]:
x <- trimws(as.character(manual_ordered_joined_df$Formula_pick))

data.frame(
  Rep_TF_for_RNA = x,
  exact_dict_match = x %in% names(gene_dict1_vec),
  uppercase_dict_match =
    toupper(x) %in% toupper(names(gene_dict1_vec))
) |>
  subset(uppercase_dict_match & !exact_dict_match)

In [ ]:
gene_dict1_vec["TEAD X2 (TANDEM)"]
# TEAD X2 (TANDEM)
#            "TEAD4"

gene_dict1_vec["SRF / CARG"]
# SRF / CARG
#      "SRF"

gene_dict1_vec["ETS::NF-KB"]
# ETS::NF-KB
#     "RELA"

In [ ]:
length(manual_ordered_df$Formula_pick)

In [ ]:
head(manual_ordered_df)

In [ ]:
desired_order

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tibble)
})

tf_vec <- manual_ordered_joined_df$Formula_pick

driver <- tibble(
  Rep_TF_for_RNA = tf_vec,
  gene_upper  = toupper(tf_vec),
  .ord = seq_along(tf_vec)
)

tpm_filt_match1 <- as.data.frame(tpm_avg) %>%
  mutate(gene_upper = toupper(gene)) %>%
  inner_join(driver, by = "gene_upper") %>%     # duplicates preserved, missing dropped
  arrange(.ord) %>%
  mutate(gene = Rep_TF_for_RNA) %>%                # label output using manual TF name
  select(gene, all_of(desired_order))           # column order

nrow(tpm_filt_match1)

In [ ]:
head(tpm_filt_match1)

In [ ]:
desired_order

In [ ]:
manual_ordered_df$cluster

In [ ]:
manual_ordered_df$cluster %>%
  unique() %>%
  as.character()

In [ ]:
manual_ordered_joined_df$Formula_pick

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(ggplot2)
})


# ============================================================
# 1. Get TF/gene order and cluster information
# ============================================================

tf_vec <- manual_ordered_joined_df$Formula_pick
cluster_vec <- manual_ordered_joined_df$cluster

driver0 <- tibble(
  Rep_TF_for_RNA = tf_vec,
  gene_upper = toupper(tf_vec),
  cluster = cluster_vec,
  .ord = seq_along(tf_vec)
)


# ============================================================
# 2. Keep TFs/genes present in TPM data
#    and preserve manual ordering
# ============================================================

tpm_filt_match1 <- as.data.frame(tpm_avg) %>%
  mutate(
    gene_upper = toupper(gene)
  ) %>%
  inner_join(
    driver0,
    by = "gene_upper"
  ) %>%
  arrange(.ord) %>%
  mutate(
    gene = Rep_TF_for_RNA
  ) %>%
  select(
    gene,
    cluster,
    .ord,
    all_of(desired_order)
  )

nrow(tpm_filt_match1)


# ============================================================
# 3. Generate unique display names for duplicated TFs
# ============================================================

gene_list_ordered <- as.character(tpm_filt_match1$gene)
cluster_list_ordered <- tpm_filt_match1$cluster
ord_list_ordered <- tpm_filt_match1$.ord

gene_labels <- make.unique(
  gene_list_ordered,
  sep = " ("
)

gene_labels <- ifelse(
  grepl("\\(\\d+$", gene_labels),
  paste0(gene_labels, ")"),
  gene_labels
)


driver <- tibble(
  gene_out = gene_list_ordered,
  gene_disp = gene_labels,
  gene_upper = toupper(gene_list_ordered),
  cluster = cluster_list_ordered,
  .ord = ord_list_ordered
)


# ============================================================
# 4. Average TPM across replicates
# ============================================================

tpm_by_sample <- as.data.frame(tpm) %>%
  rownames_to_column("gene") %>%
  pivot_longer(
    cols = -gene,
    names_to = "col",
    values_to = "TPM"
  ) %>%
  left_join(
    rep_map,
    by = "col"
  ) %>%
  drop_na(sample) %>%
  group_by(gene, sample) %>%
  summarize(
    mean_TPM = mean(TPM, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  filter(
    is.finite(mean_TPM),
    mean_TPM >= 0
  )


# ============================================================
# 5. Match selected TFs and calculate log2(TPM + 0.1)
#
#    Color:
#      within-gene z-score calculated from log2(TPM + 0.1)
#
#    Size:
#      log2(TPM + 0.1)
#
#    No thresholding or clipping is applied.
# ============================================================

tpm_plot <- tpm_by_sample %>%
  mutate(
    gene_upper = toupper(gene)
  ) %>%
  inner_join(
    driver,
    by = "gene_upper"
  ) %>%
  arrange(.ord) %>%
  mutate(
    gene = gene_disp,

    # log2 transformation with pseudocount = 0.1
    log2_TPM = log2(mean_TPM + 0.1)
  ) %>%
  group_by(gene) %>%
  mutate(

    mean_log2 = mean(
      log2_TPM,
      na.rm = TRUE
    ),

    sd_log2 = sd(
      log2_TPM,
      na.rm = TRUE
    ),

    # Within-gene z-score
    z = ifelse(
      is.finite(sd_log2) & sd_log2 > 0,
      (log2_TPM - mean_log2) / sd_log2,
      0
    )
  ) %>%
  ungroup()


# ============================================================
# 6. Sample ordering
# ============================================================

if (exists("desired_order")) {

  present <- desired_order[
    desired_order %in% unique(tpm_plot$sample)
  ]

  if (length(present) > 0) {

    tpm_plot <- tpm_plot %>%
      mutate(
        sample = factor(
          sample,
          levels = present
        )
      )
  }
}


# ============================================================
# 7. Gene and cluster ordering
# ============================================================

gene_levels <- rev(driver$gene_disp)

cluster_levels <- manual_ordered_joined_df$cluster %>%
  unique() %>%
  as.character()

tpm_plot <- tpm_plot %>%
  mutate(

    gene = factor(
      gene,
      levels = gene_levels
    ),

    cluster = factor(
      as.character(cluster),
      levels = cluster_levels
    )
  )


# ============================================================
# 8. Define sample blocks
# ============================================================

sample_block_df <- tibble(
  sample = desired_order,

  sample_block = c(
    rep("MEF", 7),
    rep("NSC", 5),
    rep("ESC", 3),
    rep("Bcell", 4)
  )
)

stopifnot(
  nrow(sample_block_df) == length(desired_order)
)


# Add sample-block information
tpm_plot <- tpm_plot %>%
  mutate(
    sample = as.character(sample)
  ) %>%
  left_join(
    sample_block_df,
    by = "sample"
  ) %>%
  mutate(

    sample = factor(
      sample,
      levels = desired_order
    ),

    sample_block = factor(
      sample_block,
      levels = c(
        "MEF",
        "NSC",
        "ESC",
        "Bcell"
      )
    )
  )


# ============================================================
# 9. Plot
#
#    Fill = z-score of log2(TPM + 0.1)
#
#    Size = log2(TPM + 0.1)
#
#    Radius scaling is used to make low-expression values
#    visually smaller while preserving the full continuous
#    range and without applying thresholds.
#
#    Five reference circle sizes:
#      0, 1, 2, 4, 8
# ============================================================

p_tpm_dot <- ggplot(
  tpm_plot,
  aes(
    x = sample,
    y = gene
  )
) +

  geom_point(
    aes(
      fill = z,
      size = log2_TPM
    ),
    shape = 21,
    color = "black",
    stroke = 0.20,
    alpha = 0.9
  ) +

  # ----------------------------------------------------------
  # Color: within-gene z-score
  # ----------------------------------------------------------

  scale_fill_gradient2(
    low = "navy",
    mid = "grey95",
    high = "firebrick3",
    midpoint = 0,
    name = "Within gene z score"
  ) +

  # ----------------------------------------------------------
  # Size: log2(TPM + 0.1)
  #
  # scale_radius() scales point radius directly.
  #
  # Full original expression range is retained:
  #
  #   TPM = 0
  #   log2(0 + 0.1) = -3.321928
  #
  # Therefore no thresholding/clipping is introduced.
  #
  # Five legend reference values:
  #   0, 1, 2, 4, 8
  # ----------------------------------------------------------

  scale_radius(
    limits = c(
      log2(0.1),
      NA
    ),
    breaks = c(
      0,
      1,
      2,
      4,
      8
    ),
    range = c(
      0,
      6
    ),
    name = "log2(TPM + 0.1)"
  ) +

  # ----------------------------------------------------------
  # Labels
  # ----------------------------------------------------------

  labs(
    x = "Sample",
    y = "Gene",
    title = "Gene expression across samples (TPM, replicate averaged)"
  ) +

  # ----------------------------------------------------------
  # Separate TF clusters and sample blocks
  # ----------------------------------------------------------

  facet_grid(
    rows = vars(cluster),
    cols = vars(sample_block),
    scales = "free",
    space = "free",
    switch = "y"
  ) +

  # ----------------------------------------------------------
  # Theme
  # ----------------------------------------------------------

  theme_bw(
    base_size = 11
  ) +

  theme(

    panel.grid = element_blank(),

    panel.spacing.y = grid::unit(
      1,
      "mm"
    ),

    panel.spacing.x = grid::unit(
      2,
      "mm"
    ),

    axis.text.x = element_text(
      angle = 90,
      vjust = 0.5,
      hjust = 1
    ),

    axis.text.y = element_text(
      size = 9
    ),

    plot.title = element_text(
      hjust = 0.5,
      size = 13,
      face = "bold"
    ),

    strip.placement = "outside",

    strip.background = element_blank(),

    strip.text.y.left = element_text(
      angle = 0,
      face = "bold",
      size = 10
    ),

    strip.text.x = element_text(
      face = "bold",
      size = 10
    )
  )


# ============================================================
# 10. Save figure
# ============================================================

outfile <- file.path(
  "Motif_TFs_genes_dotplot_log2TPM_plus0.1_size_z_cluster_V10.pdf"
)

ggsave(
  filename = outfile,
  plot = p_tpm_dot,
  width = 6.5,
  height = 15,
  dpi = 300
)

message(
  "Saved: ",
  outfile
)


# ============================================================
# 11. Display plot
# ============================================================

p_tpm_dot

In [ ]:
tpm_plot %>%
  filter(
    as.character(gene) %in% c("POU5F1", "SOX2","HOXA9"),
    # sample_block == "MEF"
  ) %>%
  select(
    gene,
    sample,
    mean_TPM,
    log2_TPM,
    z
  )

In [ ]:
levels(tpm_plot$sample)

In [ ]:
# tpm_filt_match1